<h1> Eval </h1>

In [1]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, BitsAndBytesConfig

from datasets import load_dataset 

import torch

In [2]:
benchmark = load_dataset ("inesc-id/camoes_SI", split = "test", streaming = True)#.take(50) #https://huggingface.co/docs/datasets/stream

In [3]:
print (benchmark)

exemplo = next(iter(benchmark))

print (exemplo)

benchmark = benchmark.take(50) #aponta para um iterável que limita a leitura aos primeiros 3000 exemplos

IterableDataset({
    features: ['audio', 'age', 'gender', 'hypothesis', 'reference', 'speaker_id', 'wrd', 'wer', 'dataset', 'ID', 'sex'],
    n_shards: 3
})
{'audio': {'path': None, 'array': array([0.00521851, 0.00534058, 0.00509644, ..., 0.00695801, 0.0062561 ,
       0.00512695], shape=(40112,)), 'sampling_rate': 16000}, 'age': '16.0', 'gender': 'nan', 'hypothesis': 'dezasseis anos desde desde que nasci', 'reference': 'dezesseis anos desde desde que nasci', 'speaker_id': 'Fal03', 'wrd': 'dezesseis anos desde desde que nasci', 'wer': '16.67', 'dataset': 'FBracarense', 'ID': 'nan', 'sex': 'nan'}


<hr>

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print (device)

cuda


In [6]:
MODELO_PATH = r"C:\Users\Admin\Desktop\models\SPEECH AI\SpeechAI-8Bit"


try:
    PROCESSADOR =  AutoProcessor.from_pretrained (MODELO_PATH)
    WHISPER = AutoModelForSpeechSeq2Seq.from_pretrained (MODELO_PATH, device_map = device)


except Exception as e:
    print (e)

W0730 10:35:49.906000 4280 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 1259/1259 [00:01<00:00, 645.26it/s]


<h1> Quant </h1>

In [ ]:
MODELO_NAME = "inesc-id/WhisperLv3-PT-All"

quantization = BitsAndBytesConfig (
    load_in_4bit = True,  #Quantização
    bnb_4bit_quant_type = 'nf4',  #Tipo de Quantização
    bnb_4bit_use_double_quant = True, #Double Quantization
    bnb_4bit_compute_dtype = 'float16', #Tipo de precisão usada nos cálculos durante a inferência.
)

try:
    PROCESSADOR = AutoProcessor.from_pretrained (MODELO_NAME) #Tokenizer Wanna Be
    MODELO_SPEECH = AutoModelForSpeechSeq2Seq.from_pretrained (MODELO_NAME, device_map = device, quantization_config = quantization)


    PROCESSADOR.save_pretrained (r"C:\Users\Admin\Desktop\models\SPEECH AI\SpeechAI")
    MODELO_SPEECH.save_pretrained (r"C:\Users\Admin\Desktop\models\SPEECH AI\SpeechAI")


except Exception as e:
    print (e)


In [7]:
print (torch.cuda.memory_allocated() / 1024**3, "GB")
print (torch.cuda.memory_reserved() / 1024**3, "GB")

'''torch.cuda.empty_cache()
torch.cuda.reset_max_memory_allocated()
del PROCESSADOR, MODELO_SPEECH'''

print (torch.cuda.memory_allocated() / 1024**3, "GB")
print (torch.cuda.memory_reserved() / 1024**3, "GB")



1.6517448425292969 GB
1.658203125 GB
1.6517448425292969 GB
1.658203125 GB


<h1> Main </h1>

In [ ]:
ground_truth = []

asr = []

with torch.no_grad():
    for x in benchmark:

        ground_truth.append (x["reference"])

        wav = x["audio"]["array"]

        inputs = PROCESSADOR (wav, sampling_rate = x["audio"]["sampling_rate"], return_tensors = "pt").to(device)

        outputs = WHISPER.generate (**inputs)

        asr.append (PROCESSADOR.decode (outputs, skip_special_tokens = True))

        print (len(ground_truth))

In [11]:
print (ground_truth)
print (asr)


['dezesseis anos desde desde que nasci', 'e pronto livre indireto é se alguém der mão ou assim é que é indireto', 'livre indireto ou direto direto é pode chutar', 'sem bater em ninguém', 'livre indireto é', 'ou alguém toca para à frente e ele chuta ou tem que haver dois toques pronto', 'gosto de quase tudo todo o tipo de desportos', 'a diferença é muita', 'campo de futebol de cinco', 'duas balizas o objetivo é o mesmo', 'e depois pronto os lançamentos', 'podem ser com o pé', 'não há atrasos', 'à sexta falta é livre direto', 'é de nove metros', 'isto é', 'se os jogadores fizerem seis faltas', 'existe lá um ponto nove metros da baliza até', 'até onde chegar os nove metros no meio', 'e à um jogador que vai bater', 'vai marcar é como se fosse um penálti', 'e chuta e pronto', 'se for golo foi se não for continua o jogo', 'os livres', 'para o para o adversário e chuta e logo se vê', 'é pontapé de baliza', 'quando um jogador está a fazer um passe', 'ou se não for o guardaredes tem que ser out

In [27]:
from jiwer import wer
import jiwer

x = str(asr[0])

print (x.lower)

<built-in method lower of str object at 0x000002061A24E4B0>


In [30]:
y = []
z = []

for x in asr:

    ex = x[0].lower()
    ex = jiwer.RemovePunctuation()(ex)
    ex = jiwer.Strip()(ex)

    y.append(ex)



for h in ground_truth:

    ex = h.lower()
    ex = jiwer.RemovePunctuation()(ex)
    ex = jiwer.Strip()(ex)
    
    z.append(ex)

print (y)
print (z)

erro = wer (y, z)

print (erro)

['você sabe desde que nasce', 'e pronto liberta a salguém da irmão você é direito', 'livering direct o dirta o direito pode chutar', '', 'livre em direito é a', 'logan toca para frente e ele show tem cabelos por frente', 'gosto de quatro tipos de', 'a diferência é muito', 'comfortable sympathing', 'duas bolezas o objetivo é o mesmo', 'después planteando os lançamentos', 'podem ser culpa aí', 'não atrasos', 'a sextafalta é a libre dire', 'ermetrometrometrometro', '一人は', 'seus jogadores fizeram essas faltas', 'existe lá um ponto não metros da baleza até', '', 'žada', 'vai marcar a que nos pensa penal', '', 'foi o gol foi se não for continuou', 'os libros diretores', 'padusand iš tai', '', 'quando uma esgota está fazendo um passe', 'ou se não for agora a resto tem que ser outro', '', 'depois da linha dos jogadores', 'eu sei o que é', 'saportar um jogador e se o jogador pensar', 'portanto quatro centrais', 'e dois laterais pronto', 'seem with linha', 'no', 'e um jogador tá que me tava que'